In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(1337)

device = torch.device('cuda' if torch.cuda.is_available() 
         else 'mps' if torch.backends.mps.is_available() 
         else 'cpu')
device

device(type='cuda')

In [3]:
with open('../data/input.txt', 'r', encoding='utf-8') as f:
    text = f.read()
len(text)

1115393

In [4]:
vocab = sorted(list(set(text)))
vocab_size = len(vocab)
vocab_size

65

In [5]:
stoi = {ch:i for i, ch in enumerate(vocab)}
itos = {i:ch for i, ch in enumerate(vocab)}
encode = lambda s: [stoi[ch] for ch in s]
decode = lambda l: ''.join([itos[i] for i in l])
decode(encode('hi there'))

'hi there'

In [6]:
data = torch.tensor(encode(text), dtype=torch.long)
n = (int)(len(data) * 0.9)
train_data = data[:n]
val_data = data[n:]
len(data) == len(train_data) + len(val_data)

True

In [7]:
block_size = 256
batch_size = 64
max_iterations = 5000
eval_iterations = 200
eval_interval = 500
learning_rate = 3e-4
n_embed = 384
n_head = 6
n_layers = 6
dropout = 0.2

In [8]:
def get_batch(split):
    data = train_data if split == 'train' else val_data
    idx = torch.randint(len(data)-block_size, (batch_size,))
    xb = torch.stack([data[i:i+block_size] for i in idx])
    yb = torch.stack([data[i+1:i+block_size+1] for i in idx])
    x, y = xb.to(device), yb.to(device)
    return x,y

In [9]:
x, y = get_batch('train')
x, y

(tensor([[ 1, 56, 53,  ..., 53, 51,  1],
         [10,  1, 39,  ..., 46, 43,  1],
         [50,  1, 58,  ..., 42,  1, 51],
         ...,
         [51, 43, 50,  ..., 43,  1, 46],
         [ 0, 35, 47,  ..., 54, 53, 47],
         [57, 59, 56,  ..., 45, 57,  1]], device='cuda:0'),
 tensor([[56, 53, 63,  ..., 51,  1, 32],
         [ 1, 39,  1,  ..., 43,  1, 52],
         [ 1, 58, 43,  ...,  1, 51, 39],
         ...,
         [43, 50, 43,  ...,  1, 46, 43],
         [35, 47, 50,  ..., 53, 47, 50],
         [59, 56, 43,  ..., 57,  1, 53]], device='cuda:0'))

In [10]:
class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads):
        super().__init__()
        dim_head = n_embed // num_heads
        self.multiheads = nn.ModuleList([SingleHeadAttention(dim_head) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embed, n_embed)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        out = torch.cat([h(x) for h in self.multiheads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

In [11]:
class FeedForward(nn.Module):
    def __init__(self):
        super().__init__()
        self.ffw = nn.Sequential(
            nn.Linear(n_embed, 4 * n_embed),
            nn.ReLU(),   
            nn.Linear(4 * n_embed, n_embed), 
            nn.Dropout(dropout)        
        )

    def forward(self, x):
        out = self.ffw(x)
        return out

In [12]:
class Block(nn.Module):
    def __init__(self, num_heads):
        super().__init__()
        self.ffwd = FeedForward()
        self.ma = MultiHeadAttention(num_heads)
        self.ln1 = nn.LayerNorm(n_embed)
        self.ln2 = nn.LayerNorm(n_embed)

    def forward(self, x):
        x = x + self.ma(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

In [13]:
class SingleHeadAttention(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embed, head_size, bias=False) #E,H
        self.query = nn.Linear(n_embed, head_size, bias=False)
        self.value = nn.Linear(n_embed, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x):
        B,T,E = x.shape
        k = self.key(x) #B,T,E @ E,H -> B,T,H
        q = self.query(x)
        v = self.value(x)
        
        wei = k @ q.transpose(-2, -1) * E ** -0.5 #B,T,H @ B,H,T -> B,T,T
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))

        wei = wei.softmax(dim=-1)       
        wei = self.dropout(wei)
         
        out = wei @ v #B,T,T @ B,T,H -> B,T,H
        return out

In [14]:
class Bitsy(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embed)
        self.pos_embedding_table = nn.Embedding(block_size, n_embed)
        self.blocks = nn.Sequential(*[Block(n_head) for _ in range(n_layers)])
        self.ln = nn.LayerNorm(n_embed)
        self.lm_head = nn.Linear(n_embed, vocab_size)

    def forward(self, idx, targets=None):
        B,T = idx.shape
        tok_emb = self.token_embedding_table(idx) #BTV
        pos_emb = self.pos_embedding_table(torch.arange(T, device=device))
        x = tok_emb + pos_emb
        x = self.blocks(x) #BTH == BTE
        x = self.ln(x)
        logits = self.lm_head(x) #BTE @ EV => BTV
        
        if targets == None:
            loss = None
        else:
            B,T,C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)        
        return logits, loss

    @torch.no_grad()
    def evaluate(self):
        out = {}
        self.eval()
        for split in ['train', 'val']:
            losses = torch.zeros(eval_iterations)
            for k in range(eval_iterations):
                x, y = get_batch(split)
                logits, loss = self(x,y)
                losses[k] = loss.item()
            out[split] = losses.mean()
        self.train()
        return out

    @torch.no_grad()
    def generate(self, idx, max_tokens=300):
        for _ in range(max_tokens):
            idx_cond = idx[:, -block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=-1)
        return idx

In [15]:
model = Bitsy()
m = model.to(device)
optimizer = torch.optim.AdamW(m.parameters(), lr=learning_rate)
m


Bitsy(
  (token_embedding_table): Embedding(65, 384)
  (pos_embedding_table): Embedding(256, 384)
  (blocks): Sequential(
    (0): Block(
      (ffwd): FeedForward(
        (ffw): Sequential(
          (0): Linear(in_features=384, out_features=1536, bias=True)
          (1): ReLU()
          (2): Linear(in_features=1536, out_features=384, bias=True)
          (3): Dropout(p=0.2, inplace=False)
        )
      )
      (ma): MultiHeadAttention(
        (multiheads): ModuleList(
          (0-5): 6 x SingleHeadAttention(
            (key): Linear(in_features=384, out_features=64, bias=False)
            (query): Linear(in_features=384, out_features=64, bias=False)
            (value): Linear(in_features=384, out_features=64, bias=False)
            (dropout): Dropout(p=0.2, inplace=False)
          )
        )
        (proj): Linear(in_features=384, out_features=384, bias=True)
        (dropout): Dropout(p=0.2, inplace=False)
      )
      (ln1): LayerNorm((384,), eps=1e-05, elementwise_af

In [ ]:

#training loop
for k in range(max_iterations):
    if k % eval_interval == 0:
        losses = model.evaluate()
        print(f"step: {k}, training_loss: {losses['train']:.4f}, val_loss: {losses['val']:.4f}")
    xb, yb = get_batch('train')
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
print(f"step: {k}, training_loss: {losses['train']:.4f}, val_loss: {losses['val']:.4f}")


step: 0, training_loss: 4.3149, val_loss: 4.3077


In [ ]:
idx = torch.zeros((1,1), dtype=torch.long, device=device)
print(decode(m.generate(idx, max_tokens=1000)[0].tolist()))